In [ ]:
# imports and configuration

from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from torchao.quantization import quantize_, Int8WeightOnlyConfig

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

model_name = "/home/lecacca/VSCode Environment/BERT-Testing/bert_uncased_L-2_H-128_A-2"
texts_path = Path("/home/lecacca/VSCode Environment/BERT-Testing/visualization/scatter_plot_torchao/masked_texts.txt")

block_index = 0

sublayers = [
    "attention.self.query",
    "attention.self.key",
    "attention.self.value",
    "attention.output.dense",
    "intermediate.dense",
    "output.dense",
]

In [ ]:
# load masked texts

def load_masked_texts(path):
    texts = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith("#"):
                continue
            texts.append(line)
    return texts

masked_texts = load_masked_texts(texts_path)

print(f"Loaded {len(masked_texts)} masked texts")
for i, text in enumerate(masked_texts[:5]):
    print(f"{i}: {text}")

In [ ]:
# helper functions

def get_module_by_name(root, module_path):
    module = root
    for part in module_path.split("."):
        module = getattr(module, part)
    return module


def move_inputs_to_model_device(inputs, model):
    device = next(model.parameters()).device
    return {k: v.to(device) for k, v in inputs.items()}


def get_mask_logits(model, tokenizer, text):
    inputs = tokenizer(text, return_tensors="pt")
    inputs = move_inputs_to_model_device(inputs, model)

    mask_positions = (inputs["input_ids"] == tokenizer.mask_token_id).nonzero(as_tuple=False)

    if mask_positions.numel() == 0:
        raise ValueError(f"No [MASK] token found in text: {text}")

    if mask_positions.shape[0] > 1:
        print(f"Warning: multiple [MASK] tokens found. Using first one for text: {text}")

    mask_index = mask_positions[0, 1].item()

    with torch.inference_mode():
        outputs = model(**inputs)

    logits = outputs.logits
    mask_logits = logits[0, mask_index, :].detach().float().cpu()

    return mask_logits


def get_top1_token_from_logits(logits, tokenizer):
    top_idx = torch.argmax(logits).item()
    return tokenizer.convert_ids_to_tokens([top_idx])[0]


def quantize_linear_weights_per_channel_manual(layer, num_bits=8, verbose=False):
    """
    Manual fake INT8 per-output-channel quantization.

    This approximates TorchAO Int8WeightOnlyConfig behavior for Linear weights:
    - weight-only
    - per output channel/row
    - symmetric zero point
    - dequantized back into FP32 weights

    For nn.Linear, weight shape is [out_features, in_features].
    """

    with torch.no_grad():
        weights = layer.weight.data

        if weights.ndim != 2:
            raise ValueError(f"Expected 2D Linear weight, got shape {weights.shape}")

        qmin = -(2 ** (num_bits - 1))
        qmax = 2 ** (num_bits - 1) - 1

        # One scale per output row/channel.
        max_vals = weights.abs().amax(dim=1, keepdim=True)

        # Symmetric affine-style scale.
        scales = 2.0 * max_vals / float(qmax - qmin)
        scales = torch.where(scales == 0, torch.ones_like(scales), scales)

        zero_points = torch.zeros_like(scales)

        quantized = torch.round(weights / scales) + zero_points
        quantized = torch.clamp(quantized, qmin, qmax)

        dequantized = (quantized - zero_points) * scales

        layer.weight.data.copy_(dequantized)

        if verbose:
            print("\nManual per-channel fake quantization")
            print("Weight shape:", tuple(weights.shape))
            print("Scale shape:", tuple(scales.shape))
            print("qmin/qmax:", qmin, qmax)
            print("Quantized int min/max:", quantized.min().item(), quantized.max().item())

        return {
            "quantized_int": quantized.detach().clone(),
            "dequantized_weight": dequantized.detach().clone(),
            "scales": scales.detach().clone(),
            "zero_points": zero_points.detach().clone(),
            "qmin": qmin,
            "qmax": qmax,
        }


def quantize_one_sublayer_manual(model, block_index, sub_layer, verbose=False):
    layer = get_module_by_name(model.bert.encoder.layer[block_index], sub_layer)
    info = quantize_linear_weights_per_channel_manual(layer, num_bits=8, verbose=verbose)
    return info


def quantize_one_sublayer_torchao(model, block_index, sub_layer):
    layer = get_module_by_name(model.bert.encoder.layer[block_index], sub_layer)
    quantize_(layer, Int8WeightOnlyConfig())
    return layer

In [ ]:
# precompute FP32 baseline logits

tokenizer = AutoTokenizer.from_pretrained(model_name)

baseline_model = AutoModelForMaskedLM.from_pretrained(model_name).to(device)
baseline_model.eval()

baseline_logits_by_text = {}
baseline_top1_by_text = {}

for text_index, text in enumerate(masked_texts):
    logits = get_mask_logits(baseline_model, tokenizer, text)
    baseline_logits_by_text[text_index] = logits
    baseline_top1_by_text[text_index] = get_top1_token_from_logits(logits, tokenizer)

print("Finished baseline inference.")
print("Example baseline top-1 predictions:")
for i in range(min(5, len(masked_texts))):
    print(i, masked_texts[i], "->", baseline_top1_by_text[i])

In [ ]:
# run manual and TorchAO quantization experiments

results = []

for sub_layer in sublayers:
    print(f"\n=== Sublayer: block {block_index}.{sub_layer} ===")

    # Fresh model for manual fake quantization
    manual_model = AutoModelForMaskedLM.from_pretrained(model_name).to(device)
    manual_model.eval()

    quantize_one_sublayer_manual(
        manual_model,
        block_index=block_index,
        sub_layer=sub_layer,
        verbose=True
    )

    # Fresh model for TorchAO quantization
    torchao_model = AutoModelForMaskedLM.from_pretrained(model_name).to(device)
    torchao_model.eval()

    quantize_one_sublayer_torchao(
        torchao_model,
        block_index=block_index,
        sub_layer=sub_layer
    )

    for text_index, text in enumerate(masked_texts):
        baseline_logits = baseline_logits_by_text[text_index]

        manual_logits = get_mask_logits(manual_model, tokenizer, text)
        torchao_logits = get_mask_logits(torchao_model, tokenizer, text)

        manual_abs_diff = (manual_logits - baseline_logits).abs()
        torchao_abs_diff = (torchao_logits - baseline_logits).abs()

        manual_mean_abs_diff = manual_abs_diff.mean().item()
        manual_max_abs_diff = manual_abs_diff.max().item()

        torchao_mean_abs_diff = torchao_abs_diff.mean().item()
        torchao_max_abs_diff = torchao_abs_diff.max().item()

        manual_top1 = get_top1_token_from_logits(manual_logits, tokenizer)
        torchao_top1 = get_top1_token_from_logits(torchao_logits, tokenizer)
        baseline_top1 = baseline_top1_by_text[text_index]

        results.append({
            "block_index": block_index,
            "sub_layer": sub_layer,
            "text_index": text_index,
            "text": text,

            "baseline_top1": baseline_top1,
            "manual_top1": manual_top1,
            "torchao_top1": torchao_top1,

            "manual_top1_match": manual_top1 == baseline_top1,
            "torchao_top1_match": torchao_top1 == baseline_top1,

            "manual_mean_abs_logit_diff": manual_mean_abs_diff,
            "torchao_mean_abs_logit_diff": torchao_mean_abs_diff,

            "manual_max_abs_logit_diff": manual_max_abs_diff,
            "torchao_max_abs_logit_diff": torchao_max_abs_diff,
        })

    del manual_model
    del torchao_model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

df = pd.DataFrame(results)

print("Finished experiments.")
df.head()

import os

out_dir = Path(
    os.environ.get(
        "RESULTS_DIR",
        "/home/lecacca/VSCode Environment/BERT-Testing/visualization/scatter_plot_torchao/results"
    )
)

out_dir.mkdir(parents=True, exist_ok=True)

out_csv = out_dir / "manual_vs_torchao_results.csv"
df.to_csv(out_csv, index=False)

print("Saved:", out_csv)

In [ ]:
# scatter plot mean absolute logit difference

plt.figure(figsize=(9, 7))

sns.scatterplot(
    data=df,
    x="manual_mean_abs_logit_diff",
    y="torchao_mean_abs_logit_diff",
    hue="sub_layer",
    style="sub_layer",
    s=80,
    alpha=0.85,
)

max_val = max(
    df["manual_mean_abs_logit_diff"].max(),
    df["torchao_mean_abs_logit_diff"].max()
)

plt.plot([0, max_val], [0, max_val], "k--", linewidth=1, label="y = x")

plt.xlabel("Manual fake INT8 mean absolute logit diff")
plt.ylabel("TorchAO INT8 mean absolute logit diff")
plt.title(f"Manual vs TorchAO INT8 Weight-Only Quantization\nBlock {block_index}, mean absolute [MASK] logit drift")
plt.grid(True, alpha=0.3)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.savefig("manual_vs_torchao_mean_logit_drift.pdf")
plt.show()

In [ ]:
# scatter plot max absolute logit difference

plt.figure(figsize=(9, 7))

sns.scatterplot(
    data=df,
    x="manual_max_abs_logit_diff",
    y="torchao_max_abs_logit_diff",
    hue="sub_layer",
    style="sub_layer",
    s=80,
    alpha=0.85,
)

max_val = max(
    df["manual_max_abs_logit_diff"].max(),
    df["torchao_max_abs_logit_diff"].max()
)

plt.plot([0, max_val], [0, max_val], "k--", linewidth=1, label="y = x")

plt.xlabel("Manual fake INT8 max absolute logit diff")
plt.ylabel("TorchAO INT8 max absolute logit diff")
plt.title(f"Manual vs TorchAO INT8 Weight-Only Quantization\nBlock {block_index}, max absolute [MASK] logit drift")
plt.grid(True, alpha=0.3)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.savefig("manual_vs_torchao_max_logit_drift.pdf")
plt.show()

In [ ]:
# scatter plot by sublayer

g = sns.FacetGrid(
    df,
    col="sub_layer",
    col_wrap=3,
    height=4,
    sharex=False,
    sharey=False
)

g.map_dataframe(
    sns.scatterplot,
    x="manual_mean_abs_logit_diff",
    y="torchao_mean_abs_logit_diff",
    s=70,
    alpha=0.85
)

for ax in g.axes.flatten():
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    max_val = max(xlim[1], ylim[1])
    ax.plot([0, max_val], [0, max_val], "k--", linewidth=1)
    ax.grid(True, alpha=0.3)

g.set_axis_labels(
    "Manual mean abs logit diff",
    "TorchAO mean abs logit diff"
)

g.fig.suptitle(
    f"Manual vs TorchAO INT8 Quantization by Sublayer, Block {block_index}",
    y=1.03
)

plt.tight_layout()
plt.savefig("manual_vs_torchao_mean_logit_drift_subplots.pdf")
plt.show()